# Fraud Detection LightGBM Demo

This notebook loads precomputed train/validation splits and runs the LightGBM model.

- LightGBM uses a leaf-wise tree growth strategy rather than the level-wise approach used by XGBoost, which means it finds better splits faster and reaches lower loss in fewer iterations
- training speed is significantly faster than XGBoost on large tabular datasets, making hyperparameter iteration practical without GPU hardware
- the scale_pos_weight parameter gives direct control over how aggressively the model penalises missed fraud cases, which is critical in an imbalanced setting where fraud is a small fraction of all transactions
- built-in early stopping on a validation set prevents overfitting without a separate regularisation tuning pass
- feature importances based on split counts are computed automatically and are easy to interpret for stakeholders in a financial domain

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').exists():
            return p
    return start

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')


Repo root: d:\DS Capstone Project\fraud-detection-and-financial-decision-system


In [2]:
from src.fraud_detection_models.lightgbm_model import load_splits, train, evaluate, save_model

X_train, y_train, X_val, y_val = load_splits()
X_train.shape, X_val.shape, y_train.value_counts()


Loading preprocessed splits from disk...
  Converting dtypes to float32...
  X_train : (472432, 221) | fraud rate: 0.0350
  X_val   : (118108, 221)   | fraud rate: 0.0350
  Dtypes  : [dtype('float32')]


((472432, 221),
 (118108, 221),
 isFraud
 0    455902
 1     16530
 Name: count, dtype: int64)

In [3]:
model = train(X_train, y_train, X_val, y_val)
results = evaluate(model, X_val, y_val, threshold=0.2)
save_model(model)
results



Training LightGBM model...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.884872	valid_0's binary_logloss: 0.110128
[200]	valid_0's auc: 0.897029	valid_0's binary_logloss: 0.0967739
[300]	valid_0's auc: 0.907289	valid_0's binary_logloss: 0.0891277
[400]	valid_0's auc: 0.915747	valid_0's binary_logloss: 0.0841601
[500]	valid_0's auc: 0.922416	valid_0's binary_logloss: 0.0805027
[600]	valid_0's auc: 0.928013	valid_0's binary_logloss: 0.0777395
[700]	valid_0's auc: 0.933348	valid_0's binary_logloss: 0.0753813
[800]	valid_0's auc: 0.937198	valid_0's binary_logloss: 0.0734856
[900]	valid_0's auc: 0.940434	valid_0's binary_logloss: 0.0718171
[1000]	valid_0's auc: 0.943177	valid_0's binary_logloss: 0.0703187
Did not meet early stopping. Best iteration is:
[1000]	valid_0's auc: 0.943177	valid_0's binary_logloss: 0.0703187

Best iteration: 1000

LIGHTGBM EVALUATION RESULTS

Threshold : 0.2
ROC-AUC   : 0.9432   ← primary metric (target: 0.975+)
PR-AUC    : 

{'auc': 0.9431772910243083,
 'pr_auc': 0.7210327746757039,
 'recall': 0.6397290104040648,
 'precision': 0.7364902506963789,
 'f1': 0.68470801502007,
 'y_prob': array([0.00420548, 0.00972076, 0.01102767, ..., 0.01735205, 0.01830616,
        0.00475157], shape=(118108,))}